# Réplica de De Nardin et al. (WACV 2024) — segmentación de layout one-shot

Implementa *A One-Shot Learning Approach to Document Layout Segmentation of Ancient Arabic
Manuscripts* para distinguir **fondo / texto principal / nota al margen**, y **compara los
resultados contra las métricas publicadas en el paper** (Tablas 1, 3 y 4).

Datos: las 4 páginas enviadas por el docente en el
[issue #1](https://github.com/gabrielaz/Etiquetado/issues/1) — 3 de train (una por
manuscrito, setup one-shot) y 1 de test. Ver `datos-docente/README.md`.

El ground truth es **pixel-precise**: las regiones rectangulares anotadas a mano en la app
del proyecto, intersectadas con la máscara de tinta de Sauvola
(`backend/scripts/build_gt_pixel.py`). Es necesario porque el paper mide F1 sobre píxeles de
tinta, no sobre rectángulos.

**Antes de correr:**
1. En tu máquina: `cd backend && python -m scripts.build_gt_pixel --project-id 2 --out ../colab/dataset_layout_arabe`
2. Comprimí `images/` y `masks/` de `colab/dataset_layout_arabe/` en un `dataset_layout_arabe.zip` y subilo abajo.

**Salidas:** las tablas comparativas contra el paper y un `layout_segmenter.onnx` descargable.

## 0. Setup

In [ ]:
!pip install -q scikit-image onnx onnxruntime onnxscript segmentation-models-pytorchimport torch, torchvisionprint('torch', torch.__version__, '| cuda disponible:', torch.cuda.is_available())if torch.cuda.is_available():    print('GPU asignada por Colab:', torch.cuda.get_device_name(0))   # importante para la Tabla 4

## 1. Subir y descomprimir el dataset exportado

In [ ]:
from google.colab import files
import zipfile, os

uploaded = files.upload()  # selecciona dataset_layout_arabe.zip
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('dataset')

# Localiza images/ y masks/ sin importar si el zip se comprimio desde adentro o desde
# afuera de la carpeta (los dos casos dan estructuras distintas al extraer).
def buscar(nombre, raiz='dataset'):
    for dirpath, dirnames, _ in os.walk(raiz):
        if nombre in dirnames:
            return os.path.join(dirpath, nombre)
    raise FileNotFoundError(
        f"No se encontro una carpeta '{nombre}/' dentro del zip. "
        f"Comprimi images/ y masks/ estando parado dentro de colab/dataset_layout_arabe/."
    )

images_dir = buscar('images')
masks_dir = buscar('masks')
print('images_dir =', images_dir, '| masks_dir =', masks_dir)

# Los nombres de archivo son los document_id de la app. Mapeo explicito a que pagina es cada
# uno, para no depender del orden alfabetico (ver datos-docente/README.md).
DOC_IDS = {'8': 'train_1', '6': 'train_2', '7': 'train_3', '9': 'test_1'}

ids = sorted(f.split('.')[0] for f in os.listdir(images_dir))
train_ids = [i for i in ids if DOC_IDS.get(i, '').startswith('train')]
test_ids  = [i for i in ids if DOC_IDS.get(i, '').startswith('test')]

print('paginas encontradas:', {i: DOC_IDS.get(i, '?') for i in ids})
print('train:', train_ids, '| test:', test_ids)
assert len(train_ids) >= 1 and len(test_ids) >= 1, 'Faltan paginas de train o de test'

## 2. Dataset + Generación Dinámica de Instancias

Reproduce el módulo del paper: redimensiona cada página a un tamaño fijo, la divide en un
grid de parches base sin solape, y añade `k` recortes aleatorios adicionales **por época**
(no fijos entre épocas) como augmentation — es la pieza que hace viable entrenar con 1-3
imágenes. Valores del paper: imagen 1344×2016, parches 672×672, k=12.

**Split espacial sin fuga.** Con 3 páginas y ninguna página sobrante, cualquier validación
"por página" rompería el setup one-shot (1 página por manuscrito). En su lugar se reserva la
**fila inferior del grid** de cada página para validación, y se entrena solo con la parte
superior. Los recortes aleatorios también se limitan a la zona de entrenamiento, así que
train y val no comparten un solo píxel. Es una desviación respecto del paper, que sí tiene 4
páginas de validación aparte.

In [ ]:
import randomimport numpy as npfrom PIL import Imagefrom torch.utils.data import Dataset# Los encoders preentrenados en ImageNet esperan la entrada normalizada con estas# constantes. Alimentarlos con img/255 crudo (como hacia la version anterior de este# notebook) desperdicia buena parte del preentrenamiento.IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)def to_tensor(img_uint8):    """(H,W,3) uint8 -> (3,H,W) float normalizado como espera el encoder."""    x = img_uint8.astype(np.float32) / 255.0    x = (x - IMAGENET_MEAN) / IMAGENET_STD    return torch.from_numpy(x).permute(2, 0, 1)RESIZE_TO = (2016, 1344)   # (alto, ancho) -- vertical: las paginas del issue son 2808x3952 y                           # similares. Es el mismo 1344x2016 (ancho x alto) que indica el paper.PATCH_SIZE = 672RANDOM_CROPS_PER_EPOCH = 12VAL_Y0 = 1344              # y >= VAL_Y0 es zona de validacion; y < VAL_Y0 es zona de traindef load_pair(doc_id):    img = Image.open(f'{images_dir}/{doc_id}.png').convert('RGB').resize((RESIZE_TO[1], RESIZE_TO[0]))    mask = Image.open(f'{masks_dir}/{doc_id}.png').convert('L').resize((RESIZE_TO[1], RESIZE_TO[0]), Image.NEAREST)    return np.array(img), np.array(mask).astype(np.int64)   # ya son indices de clase (0/1/2)def grid_patches(img, mask, y0=0, y1=None, patch=PATCH_SIZE):    """Parches base del grid, sin solape, restringidos a la banda vertical [y0, y1)."""    h, w = mask.shape    y1 = h if y1 is None else y1    out = []    for y in range(y0, y1 - patch + 1, patch):        for x in range(0, w - patch + 1, patch):            out.append((img[y:y+patch, x:x+patch], mask[y:y+patch, x:x+patch]))    return outdef random_crops(img, mask, n=RANDOM_CROPS_PER_EPOCH, y0=0, y1=None, patch=PATCH_SIZE):    h, w = mask.shape    y1 = h if y1 is None else y1    out = []    for _ in range(n):        y = random.randint(y0, y1 - patch)        x = random.randint(0, w - patch)        out.append((img[y:y+patch, x:x+patch], mask[y:y+patch, x:x+patch]))    return outclass LayoutPatchDataset(Dataset):    """Regenera los recortes aleatorios en cada epoca (Dynamic Instance Generation).    n_random=0 desactiva la generacion dinamica -> es la config "baseline" de la ablacion.    """    def __init__(self, doc_ids, y0=0, y1=None, n_random=RANDOM_CROPS_PER_EPOCH):        self.pairs = [load_pair(d) for d in doc_ids]        self.y0, self.y1, self.n_random = y0, y1, n_random        self.regenerate()    def regenerate(self):        patches = []        for img, mask in self.pairs:            patches += grid_patches(img, mask, self.y0, self.y1)            if self.n_random:                patches += random_crops(img, mask, self.n_random, self.y0, self.y1)        self.patches = patches    def __len__(self):        return len(self.patches)    def __getitem__(self, i):        img, mask = self.patches[i]        img_t = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0        mask_t = torch.from_numpy(mask).long()        return img_t, mask_t

## 3. Datasets de train y validación

Las 3 páginas se usan **todas** para entrenar (one-shot: 1 por manuscrito). La separación
train/val es espacial, no por página.

In [ ]:
def make_datasets(n_random=RANDOM_CROPS_PER_EPOCH):
    train_ds = LayoutPatchDataset(train_ids, y0=0, y1=VAL_Y0, n_random=n_random)
    val_ds   = LayoutPatchDataset(train_ids, y0=VAL_Y0, y1=RESIZE_TO[0], n_random=0)
    return train_ds, val_ds

train_ds, val_ds = make_datasets()
print(f'{len(train_ds)} parches de train por epoca, {len(val_ds)} de val (fila inferior, fija)')

## 4. Modelo: DeepLabv3+ con Aligned Xception (el backbone del paper)El paper usa **DeepLabv3+ con Aligned XCeption**. La versión anterior de este notebook usaba`deeplabv3_resnet50` de torchvision, que es **DeepLabv3 sin decoder** — o sea la arquitecturadel trabajo *previo* de los mismos autores (WACV 2023). El aporte declarado del cambio debackbone en WACV 2024 es justamente mejorar la delineación de bordes, que es casi todo elproblema cuando el objetivo son trazos de tinta de pocos píxeles de ancho.Se usa `segmentation_models_pytorch` con el encoder `tu-xception65` de timm, que es laAligned Xception del paper original de DeepLabv3+. `output_stride=8` da features más finasque el 16 por defecto, a costa de memoria.Nota: el encoder `xception` "clásico" de smp **no** sirve acá — no soporta modo dilatado(usa pooling para downsamplear), así que DeepLabv3+ lo rechaza. Por eso el `tu-` de timm.

In [ ]:
import segmentation_models_pytorch as smpdevice = 'cuda' if torch.cuda.is_available() else 'cpu'# En orden de preferencia: Aligned Xception-65 (el del paper) y dos alternativas por si# timm no puede bajar los pesos. Se resuelve una sola vez y queda fijo para toda la corrida,# asi el baseline de la ablacion y el modelo completo usan exactamente el mismo backbone.CANDIDATOS = ['tu-xception65', 'tu-xception41', 'resnet101']def _crear(encoder):    return smp.DeepLabV3Plus(encoder_name=encoder, encoder_weights='imagenet',                             encoder_output_stride=8, in_channels=3, classes=3)ENCODER = Nonefor cand in CANDIDATOS:    try:        _m = _crear(cand).to(device).eval()        with torch.no_grad():            _out = _m(torch.zeros(1, 3, PATCH_SIZE, PATCH_SIZE, device=device))        # La salida tiene que conservar la resolucion: si no, el F1 compararia tensores        # de distinto tamano contra el GT.        assert _out.shape[-2:] == (PATCH_SIZE, PATCH_SIZE), f'salida {tuple(_out.shape[-2:])}'        ENCODER = cand        print(f'Encoder: {cand} | salida {tuple(_out.shape)} | '              f'{sum(p.numel() for p in _m.parameters())/1e6:.1f}M params | device {device}')        if cand != CANDIDATOS[0]:            print(f'  (no se pudo usar {CANDIDATOS[0]!r}, que es el backbone del paper)')        del _m, _out        break    except Exception as e:        print(f'encoder {cand!r} descartado: {str(e)[:90]}')assert ENCODER is not None, 'Ningun encoder pudo construirse'def build_model():    return _crear(ENCODER).to(device)

## 5. Pérdida: Jaccard + Dice (igual que el paper)

In [ ]:
import torch.nn.functional as F

def jaccard_dice_loss(logits, target, num_classes=3, eps=1e-6):
    probs = F.softmax(logits, dim=1)                                        # (B, C, H, W)
    target_onehot = F.one_hot(target, num_classes).permute(0, 3, 1, 2).float()  # (B, C, H, W)

    dims = (0, 2, 3)  # suma sobre batch + espacio, deja viva la dimension de clase
    inter = (probs * target_onehot).sum(dim=dims)
    union = probs.sum(dim=dims) + target_onehot.sum(dim=dims) - inter

    jaccard = 1 - (inter + eps) / (union + eps)
    dice = 1 - (2 * inter + eps) / (probs.sum(dim=dims) + target_onehot.sum(dim=dims) + eps)

    return (jaccard.mean() + dice.mean()) / 2   # promedio final sobre las 3 clases

## 6. Entrenamiento (Adam, early stopping — hiperparámetros del paper)

`lr=1e-3`, `weight_decay=1e-5`, máx. 200 épocas, corta si val no mejora en 20 épocas
consecutivas (con un buffer de 50 épocas antes de poder cortar).

Envuelto en una función para poder llamarlo dos veces en la ablación (con y sin recortes
dinámicos) sin duplicar código.

In [ ]:
from torch.utils.data import DataLoaderMAX_EPOCHS, PATIENCE, BUFFER = 200, 20, 50def entrenar(n_random=RANDOM_CROPS_PER_EPOCH, max_epochs=MAX_EPOCHS, verbose_cada=10):    train_ds, val_ds = make_datasets(n_random)    model = build_model()    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)    best_val, epochs_no_improve, best_state = float('inf'), 0, None    for epoch in range(max_epochs):        train_ds.regenerate()   # nuevos recortes aleatorios cada epoca (no-op si n_random=0)        train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)        val_loader = DataLoader(val_ds, batch_size=4)        model.train()        train_loss = 0.0        for x, y in train_loader:            x, y = x.to(device), y.to(device)            optimizer.zero_grad()            loss = jaccard_dice_loss(model(x), y)            loss.backward()            optimizer.step()            train_loss += loss.item() * x.size(0)        train_loss /= len(train_ds)        model.eval()        val_loss = 0.0        with torch.no_grad():            for x, y in val_loader:                x, y = x.to(device), y.to(device)                val_loss += jaccard_dice_loss(model(x), y).item() * x.size(0)        val_loss /= len(val_ds)        if (epoch + 1) % verbose_cada == 0:            print(f'  epoch {epoch+1:3d} | train {train_loss:.4f} | val {val_loss:.4f}')        if val_loss < best_val:            best_val, epochs_no_improve = val_loss, 0            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}        else:            epochs_no_improve += 1            if epoch > BUFFER and epochs_no_improve >= PATIENCE:                print(f'  early stopping en epoch {epoch+1}')                break    model.load_state_dict(best_state)    print(f'  mejor val loss: {best_val:.4f}')    return modelprint('Entrenando el framework completo (con Dynamic Instance Generation)...')model = entrenar()

## 7. Inferencia sobre la página completa + refinamiento Sauvola

La inferencia sigue al paper: **solo los parches base del grid**, sin recortes aleatorios.
Con `RESIZE_TO=(2016,1344)` y parches de 672 el grid es exactamente 3×2, sin resto.

Sauvola (`window_size=31`, `k=0.2`) se aplica *después* de la predicción: dentro de una
región predicha, lo que no es tinta real vuelve a fondo. El paper lo señala como su
principal limitación (esos dos valores se fijan a mano).

In [ ]:
from skimage.filters import threshold_sauvolafrom skimage.color import rgb2graydef predict_page(model, img_rgb_uint8, patch=PATCH_SIZE):    """Predice la pagina completa reensamblando el grid de parches base (como en el paper)."""    h, w = img_rgb_uint8.shape[:2]    out = np.zeros((h, w), dtype=np.uint8)    model.eval()    for y in range(0, h - patch + 1, patch):        for x in range(0, w - patch + 1, patch):            crop = img_rgb_uint8[y:y+patch, x:x+patch]            t = torch.from_numpy(crop).permute(2, 0, 1).float().unsqueeze(0) / 255.0            with torch.no_grad():                logits = model(t.to(device))['out']            out[y:y+patch, x:x+patch] = logits.argmax(dim=1)[0].cpu().numpy().astype(np.uint8)    return outdef refine_with_sauvola(img_rgb_uint8, coarse_mask, window_size=31, k=0.2):    gray = rgb2gray(img_rgb_uint8)    ink = gray < threshold_sauvola(gray, window_size=window_size, k=k)    refined = coarse_mask.copy()    refined[(coarse_mask > 0) & ~ink] = 0    return refined

## 8. Visualizar sobre la página de test

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

CMAP = ListedColormap(['black', '#3B82F6', '#F59E0B'])  # fondo / texto principal / nota al margen

test_img, test_gt = load_pair(test_ids[0])
coarse = predict_page(model, test_img)
refined = refine_with_sauvola(test_img, coarse)

fig, axes = plt.subplots(1, 4, figsize=(20, 8))
for ax, im, title in zip(axes, [test_img, test_gt, coarse, refined],
                          ['Original', 'GT (pixel-precise)', 'Predicción gruesa', 'Refinado (Sauvola)']):
    ax.imshow(im, cmap=CMAP, vmin=0, vmax=2) if im.ndim == 2 else ax.imshow(im)
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout(); plt.show()

## 9. Métrica: F1 por píxel y por clase

Es la métrica del paper (Tabla 1): F1 calculado **por píxel**, por separado para texto
principal y nota al margen, y su promedio. Se evalúa sobre la página completa reensamblada,
no sobre recortes sueltos.

```
para cada clase c:  TP = (pred==c & gt==c),  FP = (pred==c & gt!=c),  FN = (pred!=c & gt==c)
                    F1_c = 2·TP / (2·TP + FP + FN)
```

In [ ]:
CLASES = {1: 'Texto principal', 2: 'Nota al margen'}

def f1_por_clase(pred, gt):
    """F1 por pixel para cada clase + promedio macro. Devuelve dict {nombre: f1}."""
    res = {}
    for c, nombre in CLASES.items():
        tp = np.count_nonzero((pred == c) & (gt == c))
        fp = np.count_nonzero((pred == c) & (gt != c))
        fn = np.count_nonzero((pred != c) & (gt == c))
        denom = 2 * tp + fp + fn
        res[nombre] = (2 * tp / denom) if denom else float('nan')
    res['Promedio'] = np.nanmean([res[n] for n in CLASES.values()])
    return res

def evaluar(model, doc_id, con_sauvola=True):
    img, gt = load_pair(doc_id)
    pred = predict_page(model, img)
    if con_sauvola:
        pred = refine_with_sauvola(img, pred)
    return f1_por_clase(pred, gt)

resultado_final = evaluar(model, test_ids[0], con_sauvola=True)
for k, v in resultado_final.items():
    print(f'{k:18} F1 = {v:.3f}')

## 10. Estudio de ablación (Tabla 3 del paper)

Tres configuraciones. Sauvola es post-hoc, así que solo hacen falta **dos entrenamientos**:

| Config | Recortes dinámicos | Sauvola |
|---|---|---|
| Baseline | ✗ | ✗ |
| + Dynamic crops | ✓ | ✗ |
| Framework completo | ✓ | ✓ |

In [ ]:
print('Entrenando baseline (sin recortes dinamicos)...')
model_baseline = entrenar(n_random=0)

ablacion = {
    'Baseline':            evaluar(model_baseline, test_ids[0], con_sauvola=False),
    '+ Dynamic crops':     evaluar(model,          test_ids[0], con_sauvola=False),
    'Framework completo':  resultado_final,
}
for nombre, r in ablacion.items():
    print(f"{nombre:22} principal {r['Texto principal']:.3f} | marginal {r['Nota al margen']:.3f}")

## 11. Tiempos de inferencia (Tabla 4 del paper)

El paper reporta 0.41 s de segmentación gruesa + 0.01 s de Sauvola = 0.42 s por página, sobre
una **NVIDIA RTX 3090**. Colab asigna otra GPU (ver celda 0), así que los valores absolutos no
son comparables — lo que sí es comparable es la *proporción*: que el refinamiento sea
prácticamente gratis frente a la segmentación.

In [ ]:
import time

def cronometrar(model, doc_id, repeticiones=5):
    img, _ = load_pair(doc_id)
    predict_page(model, img)                       # warm-up, no se mide
    if device == 'cuda': torch.cuda.synchronize()

    t_gruesa, t_sauvola = [], []
    for _ in range(repeticiones):
        t0 = time.perf_counter()
        coarse = predict_page(model, img)
        if device == 'cuda': torch.cuda.synchronize()
        t1 = time.perf_counter()
        refine_with_sauvola(img, coarse)
        t2 = time.perf_counter()
        t_gruesa.append(t1 - t0); t_sauvola.append(t2 - t1)

    return {'Segmentación gruesa': np.mean(t_gruesa), 'Refinamiento Sauvola': np.mean(t_sauvola)}

tiempos = cronometrar(model, test_ids[0])
tiempos['Total'] = sum(tiempos.values())
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print('Hardware:', gpu)
for k, v in tiempos.items():
    print(f'{k:22} {v:.3f} s/pagina')

## 12. Comparación con las métricas publicadas del paper

Las cifras del paper están transcritas de las Tablas 1, 3 y 4 de *De Nardin et al., WACV
2024*. **Leé la celda de advertencias de abajo antes de sacar conclusiones de estos números.**

In [ ]:
import pandas as pd

# --- Tabla 1: F1 sobre el test set, contra los metodos competidores del dataset Bukhari ---
tabla1 = pd.DataFrame([
    ['Bukhari et al. [6]',           0.950, 0.950, 0.950],
    ['Barakat & El-Sana [4]',        0.950, 0.800, 0.875],
    ['Alasaam et al. [1]',           0.986, 0.969, 0.978],
    ['Droby et al. [15]',            0.986, 0.970, 0.978],
    ['Asi et al. [3]',               0.992, 0.985, 0.988],
    ['De Nardin et al. (paper)',     0.989, 0.991, 0.990],
    ['Esta réplica',
        resultado_final['Texto principal'],
        resultado_final['Nota al margen'],
        resultado_final['Promedio']],
], columns=['Método', 'Texto principal', 'Nota al margen', 'Promedio'])

# --- Tabla 3: ablacion ---
paper_ablacion = {'Baseline': (0.873, 0.933), '+ Dynamic crops': (0.953, 0.958),
                  'Framework completo': (0.989, 0.991)}
tabla3 = pd.DataFrame([
    [cfg, paper_ablacion[cfg][0], paper_ablacion[cfg][1],
     ablacion[cfg]['Texto principal'], ablacion[cfg]['Nota al margen']]
    for cfg in paper_ablacion
], columns=['Configuración', 'Paper: principal', 'Paper: marginal',
            'Réplica: principal', 'Réplica: marginal'])

# --- Tabla 4: tiempos ---
paper_tiempos = {'Segmentación gruesa': 0.41, 'Refinamiento Sauvola': 0.01, 'Total': 0.42}
tabla4 = pd.DataFrame([
    [etapa, paper_tiempos[etapa], tiempos[etapa]] for etapa in paper_tiempos
], columns=['Etapa', 'Paper (RTX 3090)', f'Réplica ({gpu})'])

for titulo, t in [('TABLA 1 — F1 vs. métodos competidores', tabla1),
                  ('TABLA 3 — Ablación', tabla3),
                  ('TABLA 4 — Tiempos de inferencia (s/página)', tabla4)]:
    print(f'\n{titulo}\n' + '=' * len(titulo))
    print(t.to_string(index=False, float_format=lambda v: f'{v:.3f}'))

print('\nMarkdown para pegar en el issue:\n')
for t in (tabla1, tabla3, tabla4):
    print(t.to_markdown(index=False, floatfmt='.3f'), '\n')

## ⚠️ Advertencias — por qué estos números NO son estrictamente comparablesLas tablas de arriba ponen las cifras lado a lado, pero **no miden lo mismo**. Cincodiferencias, todas a tener en cuenta antes de afirmar que la réplica "iguala" o "supera" al paper:1. **Test set distinto.** El paper evalúa sobre 8 páginas del dataset de Bukhari et al.; acá   hay **1 sola página**. Un F1 sobre una página no tiene intervalo de confianza útil.   (Además, los 3 adjuntos de test del issue resultaron ser el mismo archivo — ver   `datos-docente/README.md`.)2. **Ground truth distinto.** El del paper es anotación pixel-precise hecha por un experto de   dominio. El nuestro es *rectángulo anotado ∩ tinta de Sauvola*: hereda los errores de los   rectángulos, y donde marginalia y texto principal se solapan la regla es arbitraria   (marginalia gana). Entre 0.2% y 4.9% de la tinta quedó sin clase asignada.3. **Backbone distinto.** El paper usa DeepLabv3+ con Aligned XCeption; acá se usa   `deeplabv3_resnet50` de torchvision por disponibilidad directa en Colab.4. **Validación distinta.** El paper tiene 4 páginas de validación separadas; acá se reserva   la fila inferior del grid de las mismas 3 páginas de train (split espacial, sin fuga de   píxeles, pero misma distribución de página).5. **Hardware distinto.** Tabla 4: RTX 3090 en el paper vs. la GPU que asigne Colab.Los números de la Tabla 3 (ablación) son los más informativos de los tres, porque comparanconfiguraciones **entre sí bajo condiciones idénticas** — ahí sí la tendencia(baseline < +crops < completo) es una réplica legítima del hallazgo del paper.

## 13. Exportar a ONNX (para inferencia en el backend sin depender de PyTorch)El backend (`layout_ml_service.py`) usa `onnxruntime` en CPU para no arrastrar PyTorch comodependencia de producción.> ⚠️ **Incompatibilidad pendiente.** `layout_ml_service.py:51` preprocesa con `patch / 255.0`> a secas, que es lo que hacía este notebook antes. Ahora se normaliza además con la media y> el desvío de ImageNet (`to_tensor()` en la sección 2). Si vas a usar en el backend un> `.onnx` exportado desde acá, hay que aplicarle la misma normalización allá, o las> predicciones van a salir mal sin dar ningún error.

In [ ]:
model.eval()
dummy = torch.randn(1, 3, PATCH_SIZE, PATCH_SIZE, device=device)
torch.onnx.export(
    model, dummy, 'layout_segmenter.onnx',
    input_names=['image'], output_names=['logits'],
    dynamic_axes={'image': {0: 'batch', 2: 'height', 3: 'width'}, 'logits': {0: 'batch', 2: 'height', 3: 'width'}},
    opset_version=17,
    dynamo=False,
)
import os
print('Tamaño del .onnx:', os.path.getsize('layout_segmenter.onnx') / 1e6, 'MB')
files.download('layout_segmenter.onnx')
print('Descarga layout_segmenter.onnx y colócalo en backend/app/ml/layout_segmenter.onnx')

## Próximo paso en el backend (no incluido en este notebook)

1. `onnxruntime` en `backend/requirements.txt`.
2. Nueva función en `backend/app/services/layout_ml_service.py`: carga `layout_segmenter.onnx`,
   corre inferencia por parches (mismo grid `PATCH_SIZE=672`), aplica Sauvola, y separa la
   máscara de 3 clases en bboxes por región con `cv2.findContours` (uno por cada clase
   1/2), mismo formato de salida que el heurístico (`x/y/width/height/region_type/confidence`).
3. Endpoint `"ml"` en `backend/app/routers/layout.py`, patrón preview → apply.
4. Selector de método en `DLAPanel.tsx` (frontend), igual que ya existe en `LabelingPanel.tsx`.